## Boston Dataset
We are going to look at a Boston housing dataset in this example

## Imports

In [ ]:
import statsmodels.api as sm
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import numpy.typing as npt
import seaborn as sns
from sklearn import datasets
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error

In [ ]:
def rmse(y_test, y_pred):
  '''Calculates the root mean square between an array of known and predicted values'''
  return mean_squared_error(y_test, y_pred)**0.5


In [ ]:
rmse(
  np.array([1, 2, 3]),  # <= y_test
  [4, 5, 6]             # <= y_pred
)


In [ ]:
# defining a function using "type hints", which give python a hint to the data type
def rmse(y_test: npt.NDArray, y_pred: npt.NDArray) -> np.float64:
  '''Calculates the root mean square between an array of known and predicted values'''
  return np.sqrt(mean_squared_error(y_test, y_pred))


In [ ]:
rmse(
  np.array([1, 2, 3]),  # <= y_test
  [4, 5, 6]             # <= y_pred
)


In [ ]:
rmse(
  np.array([1, 2, 3]),  # <= y_test
  np.array([4, 5, 6])   # <= y_pred
)


In [ ]:
help(rmse)

## Read in Data

In [ ]:
url = "http://ddc-datascience.s3.amazonaws.com/boston.csv"
url


In [ ]:
!curl -s -I {url}


In [ ]:
!curl -s {url} | head | cut -c1-100


In [ ]:
pd.read_csv( url, header=None, skiprows=3, usecols=[1,3,5,8] ).head()

In [ ]:
boston = pd.read_csv( url, index_col = 0)
boston.head()

In [ ]:
boston.shape

## Data Cleaning

In [ ]:
boston.describe( include = "all" ).transpose()

In [ ]:
boston.info()

## EDA

In [ ]:
plt.figure(figsize = (8,6))
plt.hist(boston['med_home_value'])
plt.xlabel('Median Home Value (in thousands)')
plt.ylabel('Count')
plt.show()


In [ ]:
plt.figure(figsize = (8,6))
sns.scatterplot(x = boston['num_rooms'], y = boston['med_home_value'])
plt.xlabel('Number of Rooms')
plt.ylabel("Median Home Value (in thousands)") ;

## Linear Regression
We will first fit a multiple linear regression model using all of the predictors.

In [ ]:
# First we will break up our data into training and testing sets
X = boston.drop('med_home_value', axis = 1).copy()
X = sm.add_constant(X)
y = boston['med_home_value']

display(X.head()), display(y.head()) ;

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 4)

In [ ]:
for _ in [X_train, X_test, y_train, y_test]:
  print(type(_), _.shape)


In [ ]:
train_test_split?

In [ ]:
# Fit a linear model using Statsmodels
myfit = sm.OLS(y_train, X_train).fit()
myfit.summary()

In [ ]:
# Perform cross validation
n = 500
results = np.zeros(n)
for idx in range(n):
  X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2)
  model = sm.OLS(y_train, X_train).fit()
  y_pred = model.predict(X_test)
  results[idx] = rmse(y_test,y_pred)

print(f"CV RMSE: {results.mean().round(2)*1000}")


In [ ]:
# Generate figure of predicted vs actual results
plt.figure(figsize=(10,8));
plt.scatter(y_test, y_pred);
p1 = max(y_pred.max(), y_test.max())
plt.plot([0, p1],[0, p1], c='red')
plt.xlabel('True Values')
plt.ylabel('Predicted Values')

## Transformation of Response

Box Cox is a commonly used transformation that is used to transform a non-normal distribution to a normal distribution. Recall that one of the assumptions of linear regression is that the residuals are normally distributed. If that assumption is not being met, we can transform the response using a Box Cox transformation to try to make them more normally distributed.

In [ ]:
# QQ Plot - Residuals of Linear Model Above
import scipy.stats as stats
res = y_test - y_pred
sm.qqplot(res, fit=True, line="45")
plt.show()

In [ ]:
plt.hist(res) ;

### Box Cox Implementation

#### Box-Cox formula

$$ y({\lambda}) =
\begin{cases}
\frac{y^\lambda - 1}{\lambda} & \text{if } \lambda \neq 0 \\
\ln(y) & \text{if } \lambda = 0
\end{cases}
\,\,\,\,\,\,\text{for } y > 0 $$




Imagine the relationship between a feature and a target is non-linear, e.g. exponential.  We can use a Box-Cox transformation to linearize the relationship.

y = x ** 2

After transformation

y = log( x ** 2 )

y = 2 * log( x )







In [ ]:
from scipy import stats
from scipy import special

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state = 4)
fitted_data, fitted_lambda = stats.boxcox(y_train)
print(fitted_lambda)


In [ ]:
y_train.mean() , y_train.median()

In [ ]:
plt.hist(y_train) ;

In [ ]:
plt.hist(fitted_data) ;

In [ ]:
fitted_data.mean() , pd.Series(fitted_data).median()

## One cross validation


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2)

y_train[:].to_list()


In [ ]:
# Perform the box cox transformation on y_train
fitted_data, fitted_lambda = stats.boxcox(y_train)

fitted_lambda, fitted_data

In [ ]:
# Fit the linear regression model. Note: you can also perform box cox with Lasso or Ridge regression
model = sm.OLS(fitted_data, X_train).fit()
y_pred = model.predict(X_test)

y_pred[:].to_list()

In [ ]:
# Perform the inverse box cox to get predictions back on original scale
real_y_pred = special.inv_boxcox(y_pred, fitted_lambda)

real_y_pred.to_list()

In [ ]:
rmse(y_test,real_y_pred)


## Multiple cross validation


In [ ]:
# Perform CV
n = 500
results = np.zeros(n)
for idx in range(n):
  X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2)

  # Perform the box cox transformation on y_train
  fitted_data, fitted_lambda = stats.boxcox(y_train)

  # Fit the linear regression model. Note: you can also perform box cox with Lasso or Ridge regression
  model = sm.OLS(fitted_data, X_train).fit()
  y_pred = model.predict(X_test)

  # Perform the inverse box cox to get predictions back on original scale
  real_y_pred = special.inv_boxcox(y_pred, fitted_lambda)

  results[idx] = rmse(y_test,real_y_pred)
print(f"CV RMSE: {results.mean().round(2)*1000}")

In [ ]:
plt.hist(results) ;

In [ ]:
# QQ Plot
import scipy.stats as stats
res = y_test - real_y_pred
sm.qqplot(res, fit=True, line="45")
plt.show()

In [ ]:
plt.hist(res) ;